# Teste Random Forest (estilo curso — só XLSX)

**Sandbox Colab:** mesmos comandos da seção *Random forest* do notebook do curso (IA Expert). **Sem Parquet** — ingestão igual ao `censo_castanhal_pipeline.ipynb`: XLSX no GitHub (`censo_castanhal/censo_castanhal/`, branch `main`) via `requests` + `pd.read_excel`. Consolidação no estilo da **Célula 4** do pipeline (`df_features` = domicílios + IAH + IVS). Não substitui o notebook canônico do pipeline (Princípio V em `tasks.md`).

Ferramentas: `numpy`, `pandas`, `plotly`, `requests`, `sklearn` — como no código de referência, **sem `def`** (só laços e atribuições).

In [ ]:
!pip install -q openpyxl plotly requests "scikit-learn>=1.4"

: 

In [ ]:
import io
import os
import re
import unicodedata
from urllib.parse import quote

import numpy as np
import pandas as pd
import plotly.express as px
import requests
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

GITHUB_REPO = os.environ.get("GITHUB_REPO", "LuanLindolfo/tcc")
XLSX_BRANCH = os.environ.get("XLSX_BRANCH", "main")
GITHUB_RAW = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{XLSX_BRANCH}"
XLSX_PATH = "censo_castanhal/censo_castanhal"

ARQUIVOS = {
    "piramide_etaria": "Censo 2022 - Pirâmide etária - Castanhal (PA).xlsx",
    "razao_sexo_envelh": "Censo 2022 - Razão de sexo e índice de envelhecimento - Castanhal (PA).xlsx",
    "populacao_residente": "Censo 2022 - População residente - Municípios.xlsx",
    "densidade_demografica": "Censo 2022 - Densidade demográfica - Municípios.xlsx",
    "caract_domicilios": "Censo 2022 - Características dos domicílios - Castanhal (PA).xlsx",
    "condicoes_ocupacao": "Censo 2022 - Condições de ocupação - Castanhal (PA).xlsx",
    "tipos_domicilio": "Censo 2022 - Tipos de domicílio - Castanhal (PA).xlsx",
    "material_paredes": "Censo 2022 - Tipo de material das paredes externas - Castanhal (PA).xlsx",
    "media_moradores": "Censo 2022 - Média de moradores por domicílio - Castanhal (PA).xlsx",
    "alfabetizacao": "Censo 2022 - Alfabetização - Castanhal (PA).xlsx",
    "nivel_instrucao": "Censo 2022 - Nível de instrução - Castanhal (PA).xlsx",
    "freq_escolar": "Censo 2022 - Taxa bruta de frequência escolar, por grupo de idade - Castanhal (PA).xlsx",
    "escolaridade_homens": "analise_escolaridade_homens_percetual.xlsx",
    "escolaridade_mulheres": "analise_escolaridade_mulheres_percetual.xlsx",
    "rendimento_percapita": "Censo 2022 - Rendimento domiciliar mensal per capita - Castanhal (PA).xlsx",
    "dist_renda": "distribuição de renda.xlsx",
    "profissoes": "profissões.xlsx",
    "taxa_atividade_pct": "taxa_atividade_percentual.xlsx",
}

In [ ]:
dados_raw = {}
for chave, arquivo in ARQUIVOS.items():
    url = f"{GITHUB_RAW}/{XLSX_PATH}/{quote(arquivo)}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    dados_raw[chave] = pd.read_excel(io.BytesIO(r.content), engine="openpyxl")

dados = {}
for k, df in dados_raw.items():
    df = df.copy()
    novos = []
    for c in df.columns:
        s = unicodedata.normalize("NFKD", str(c)).encode("ascii", "ignore").decode()
        s = re.sub(r"[^\w\s]", "", s).strip().lower()
        s = re.sub(r"\s+", "_", s)
        novos.append(s)
    df.columns = novos
    num = df.select_dtypes(include="number").columns
    df[num] = df[num].fillna(df[num].median())
    dados[k] = df

frames_dem = [
    dados[k]
    for k in (
        "piramide_etaria",
        "razao_sexo_envelh",
        "populacao_residente",
        "densidade_demografica",
    )
    if k in dados and not dados[k].empty
]
df_demografico = pd.concat(frames_dem, axis=0, ignore_index=True) if frames_dem else pd.DataFrame()

frames_dom = [
    dados[k]
    for k in (
        "caract_domicilios",
        "condicoes_ocupacao",
        "tipos_domicilio",
        "material_paredes",
        "media_moradores",
    )
    if k in dados and not dados[k].empty
]
df_domicilios = pd.concat(frames_dom, axis=0, ignore_index=True) if frames_dom else pd.DataFrame()

frames_edu = [
    dados[k]
    for k in (
        "alfabetizacao",
        "nivel_instrucao",
        "freq_escolar",
        "escolaridade_homens",
        "escolaridade_mulheres",
    )
    if k in dados and not dados[k].empty
]
df_educacao = pd.concat(frames_edu, axis=0, ignore_index=True) if frames_edu else pd.DataFrame()

frames_tr = [
    dados[k]
    for k in ("rendimento_percapita", "dist_renda", "profissoes", "taxa_atividade_pct")
    if k in dados and not dados[k].empty
]
df_trabalho_renda = pd.concat(frames_tr, axis=0, ignore_index=True) if frames_tr else pd.DataFrame()

df_features = df_domicilios.copy()
cols_iah = [
    c
    for c in df_features.columns
    if any(x in c for x in ["agua", "esgoto", "lixo", "energia", "saneamento"])
]
if cols_iah:
    df_features["iah"] = (df_features[cols_iah].mean(axis=1) / 100).clip(0, 1)
else:
    df_features["iah"] = 0.5

num_cols = df_features.select_dtypes(include="number").columns.tolist()
if len(num_cols) >= 2 and len(df_features) > 6:
    X_km = MinMaxScaler().fit_transform(df_features[num_cols].fillna(0))
    km = KMeans(n_clusters=3, random_state=42, n_init=10)
    labels = km.fit_predict(X_km)
    scores = X_km.mean(axis=1)
    means = {i: scores[labels == i].mean() for i in range(3)}
    ordem = sorted(means, key=lambda k: means[k])
    nomes = {ordem[0]: "baixa", ordem[1]: "media", ordem[2]: "alta"}
    df_features["ivs_label"] = [nomes[l] for l in labels]
    df_features["cluster_ocupacao"] = labels
else:
    df_features["ivs_label"] = "media"
    df_features["cluster_ocupacao"] = 0

print(
    "shapes:",
    df_demografico.shape,
    df_domicilios.shape,
    df_educacao.shape,
    df_trabalho_renda.shape,
    df_features.shape,
)

In [ ]:
# Tabela unificada para o teste RF: todas as colunas numéricas de df_features (pipeline)
num = df_features.select_dtypes(include=["number"]).copy()
num = num.fillna(num.median(numeric_only=True))

cols = [c for c in num.columns if not str(c).startswith("_pad_rf_")]
if len(cols) < 3:
    raise ValueError("Menos de 3 colunas numéricas em df_features.")

target = "iah" if "iah" in cols else cols[2]
others = [c for c in cols if c != target]
f0, f1 = others[0], others[1]
feats = others[2:]
pad_i = 0
while len(feats) < 16:
    pad = f"_pad_rf_{pad_i}"
    num[pad] = 0.0
    feats.append(pad)
    pad_i += 1

ordered = [f0, f1, target] + feats[:16]
base_casas = num[ordered].copy()
base_casas

In [ ]:
# --- Dados no lugar de read_csv('house_prices.csv') / plano_saude2.csv — layout curso ---
base_plano_saude2 = base_casas.iloc[:, 0:2]

X_plano_saude2 = base_plano_saude2.iloc[:, 0:1].values  # previsor - : -> todos, 0:1 -> 0 até 1 (só a 0)
y_plano_saude2 = base_plano_saude2.iloc[:, 1].values  # resposta esperada - : -> todos, 1 -> coluna 1

X_casas = base_casas.iloc[:, 3:19].values  # todos os registros (:), da coluna de 3 a 19
y_casas = base_casas.iloc[:, 2].values  # todos os registros somente a coluna 2

from sklearn.model_selection import train_test_split

X_casas_treinamento, X_casas_teste, y_casas_treinamento, y_casas_teste = train_test_split(
    X_casas, y_casas, test_size=0.3, random_state=0
)

X_teste_arvore = np.arange(min(X_plano_saude2), max(X_plano_saude2), 0.1)
X_teste_arvore = X_teste_arvore.reshape(-1, 1)  # agora em matriz

In [ ]:
from sklearn.ensemble import RandomForestRegressor  # Random Forest em regressão
regressor_random_forest_saude = RandomForestRegressor(n_estimators=10)
regressor_random_forest_saude.fit(X_plano_saude2, y_plano_saude2)

In [ ]:
regressor_random_forest_saude.score(X_plano_saude2, y_plano_saude2)  # score

In [ ]:
grafico = px.scatter(x=X_plano_saude2.ravel(), y=y_plano_saude2)
grafico.add_scatter(
    x=X_teste_arvore.ravel(),
    y=regressor_random_forest_saude.predict(X_teste_arvore),
    name="Regressão",
)
grafico.show()

In [ ]:
regressor_random_forest_saude.predict([[40]])  # fazendo uma previsão com uma pessoa de 40 anos

In [ ]:
regressor_random_forest_casas = RandomForestRegressor(n_estimators=100)  # mais dados
regressor_random_forest_casas.fit(X_casas_treinamento, y_casas_treinamento)

In [ ]:
regressor_random_forest_casas.score(X_casas_treinamento, y_casas_treinamento)  # score do treinamento

In [ ]:
regressor_random_forest_casas.score(X_casas_teste, y_casas_teste)  # score do treste

In [ ]:
previsoes = regressor_random_forest_casas.predict(X_casas_teste)  # passando os dados de teste para o regressor
previsoes

In [ ]:
y_casas_teste  # valor real

In [ ]:
from sklearn.metrics import mean_absolute_error  # calculo do mean absolute error
mean_absolute_error(y_casas_teste, previsoes)